**© Copyright AIDENTIFY. All rights reserved.**

본 자료는 **멀티캠퍼스 LLM 파인튜닝 과정** 수강생을 위해 제작되었으며, 강의 목적으로만 사용 가능합니다.  
무단 복제, 배포, 상업적 이용을 금지합니다.

---

# Session 36: 프로젝트 성능 평가 및 반복 개선

## 🎯 모델 평가 및 개선

이 노트북에서는 파인튜닝된 모델의 성능을 **다각도로 평가**하고, **반복 개선 전략**을 수립합니다.

### 평가 프레임워크
```
평가 데이터 준비 → 자동 평가 → LLM-as-a-Judge → 정성 평가 → 개선 전략 → 반복 학습
```

### 평가 방법론
| 방법 | 측정 대상 | 자동화 | 신뢰도 |
|------|----------|--------|--------|
| **자동 메트릭** (ROUGE, BLEU) | 텍스트 유사도 | ✅ 완전 자동 | ⭐⭐ |
| **LLM-as-a-Judge** | 종합 품질 | ✅ 반자동 | ⭐⭐⭐⭐ |
| **정성 평가** | 실제 사용성 | ❌ 수동 | ⭐⭐⭐⭐⭐ |

### 사전 준비
- `34_project_training.ipynb`에서 저장한 파인튜닝 모델
- `eval.json` 평가 데이터
- OpenAI API 키 (LLM-as-a-Judge 사용 시)

In [ ]:
# =====================================================
# 📦 필수 패키지 임포트
# =====================================================

# !pip install rouge-score nltk bert-score openai

import torch
import json
import os
import pandas as pd
from tqdm import tqdm

print(f"🖥️ PyTorch: {torch.__version__}")
if torch.cuda.is_available():
    print(f"🎮 GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# =====================================================
# 📋 프로젝트 설정 및 모델 로드
# =====================================================

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

PROJECT_DIR = "./my_project"  # TODO: 프로젝트 디렉토리

# 프로젝트 계획 로드
with open(f"{PROJECT_DIR}/project_plan.json", "r", encoding="utf-8") as f:
    project_plan = json.load(f)

MODEL_NAME = project_plan["model_config"]["base_model"]
METHOD = project_plan["model_config"]["method"]
ADAPTER_PATH = f"{PROJECT_DIR}/models/final_model"  # TODO: 모델 경로 확인

print(f"📋 프로젝트: {project_plan['project_name']}")
print(f"🤖 베이스 모델: {MODEL_NAME}")
print(f"🔧 학습 방법: {METHOD}")

# 토크나이저 로드
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# 모델 로드
if METHOD == "QLoRA":
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
    )
    base_model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, quantization_config=bnb_config, device_map="auto", trust_remote_code=True
    )
    model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
elif METHOD == "LoRA":
    base_model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, torch_dtype=torch.float16, device_map="auto", trust_remote_code=True
    )
    model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
else:
    model = AutoModelForCausalLM.from_pretrained(
        ADAPTER_PATH, torch_dtype=torch.float16, device_map="auto", trust_remote_code=True
    )

model.eval()
print(f"\n✅ 모델 로드 완료")

---
## 1️⃣ 평가 데이터셋 준비

도메인에 맞는 테스트 셋을 준비합니다.

In [ ]:
# =====================================================
# 📂 평가 데이터 로드
# TODO: 평가 데이터 경로를 확인하세요
# =====================================================

EVAL_PATH = f"{PROJECT_DIR}/data/eval.json"  # TODO: 평가 데이터 경로

with open(EVAL_PATH, "r", encoding="utf-8") as f:
    eval_data = json.load(f)

print(f"📊 평가 데이터: {len(eval_data)}개")
print(f"\n📝 예시:")
print(json.dumps(eval_data[0], ensure_ascii=False, indent=2)[:500])

In [ ]:
# =====================================================
# 🤖 모델 응답 생성
# TODO: 시스템 프롬프트를 도메인에 맞게 수정하세요
# =====================================================

SYSTEM_MSG = "당신은 도움이 되는 AI 어시스턴트입니다."  # TODO: 수정하세요

def generate_response(model, tokenizer, prompt, system_msg="", max_new_tokens=512):
    """모델 응답을 생성합니다."""
    messages = []
    if system_msg:
        messages.append({"role": "system", "content": system_msg})
    messages.append({"role": "user", "content": prompt})
    
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            do_sample=True,
            top_p=0.9,
            repetition_penalty=1.1,
        )
    response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return response


# 전체 평가 데이터에 대해 응답 생성
print(f"🤖 {len(eval_data)}개 평가 데이터에 대해 응답 생성 중...")

predictions = []
references = []

for item in tqdm(eval_data, desc="응답 생성"):
    # 프롬프트 구성
    prompt = item["instruction"]
    if item.get("input"):
        prompt += f"\n\n{item['input']}"
    
    # 모델 응답 생성
    pred = generate_response(model, tokenizer, prompt, SYSTEM_MSG)
    predictions.append(pred)
    references.append(item["output"])

print(f"\n✅ 응답 생성 완료: {len(predictions)}개")

---
## 2️⃣ 자동 평가 메트릭 계산

### 사용 가능한 메트릭
| 메트릭 | 측정 대상 | 범위 | 높을수록 |
|--------|----------|------|----------|
| **ROUGE-1** | Unigram 겹침 | 0~1 | 좋음 |
| **ROUGE-L** | 최장 공통 부분수열 | 0~1 | 좋음 |
| **BLEU** | N-gram 정밀도 | 0~1 | 좋음 |
| **BERTScore** | 의미적 유사도 | 0~1 | 좋음 |

In [ ]:
# =====================================================
# 📊 자동 평가 메트릭 계산
# =====================================================

from rouge_score import rouge_scorer
import numpy as np

# ROUGE 계산
scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=False)

rouge_scores = {"rouge1": [], "rouge2": [], "rougeL": []}

for pred, ref in zip(predictions, references):
    scores = scorer.score(ref, pred)
    for key in rouge_scores:
        rouge_scores[key].append(scores[key].fmeasure)

# ROUGE 결과
print("📊 ROUGE Scores")
print("="*40)
for key, values in rouge_scores.items():
    print(f"  {key}: {np.mean(values):.4f} (±{np.std(values):.4f})")

In [ ]:
# =====================================================
# 📊 BLEU Score 계산
# =====================================================

import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

# nltk 데이터 다운로드 (최초 1회)
try:
    nltk.data.find("tokenizers/punkt")
except LookupError:
    nltk.download("punkt", quiet=True)
    nltk.download("punkt_tab", quiet=True)

smoother = SmoothingFunction().method1

bleu_scores = []
for pred, ref in zip(predictions, references):
    # 글자 단위 토큰화 (한국어)
    ref_tokens = list(ref)
    pred_tokens = list(pred)
    score = sentence_bleu([ref_tokens], pred_tokens, smoothing_function=smoother)
    bleu_scores.append(score)

print(f"\n📊 BLEU Score: {np.mean(bleu_scores):.4f} (±{np.std(bleu_scores):.4f})")

In [ ]:
# =====================================================
# 📊 (Optional) BERTScore 계산
# ⚠️ VRAM을 사용하므로 메모리 여유가 있을 때 실행
# =====================================================

USE_BERTSCORE = False  # TODO: BERTScore 사용 시 True

if USE_BERTSCORE:
    from bert_score import score as bert_score
    
    P, R, F1 = bert_score(
        predictions, references,
        lang="ko",
        verbose=True,
        batch_size=8
    )
    
    print(f"\n📊 BERTScore")
    print(f"  Precision: {P.mean():.4f}")
    print(f"  Recall: {R.mean():.4f}")
    print(f"  F1: {F1.mean():.4f}")
else:
    print("⏭️ BERTScore 계산을 건너뜁니다.")

---
## 3️⃣ LLM-as-a-Judge 평가

GPT-4를 활용하여 모델 출력의 품질을 **다차원으로 평가**합니다.

### 평가 차원
- **정확성 (Accuracy)**: 정보가 정확한가?
- **유용성 (Helpfulness)**: 답변이 도움이 되는가?
- **관련성 (Relevance)**: 질문에 적절히 답변하는가?
- **완결성 (Completeness)**: 충분히 상세한가?
- **안전성 (Safety)**: 유해하거나 부적절한 내용이 없는가?

In [ ]:
# =====================================================
# 🧑‍⚖️ LLM-as-a-Judge 평가기
# TODO: API 키와 평가 차원을 설정하세요
# =====================================================

from openai import OpenAI
import time

client = OpenAI(
    api_key=""  # TODO: OpenAI API 키
)

JUDGE_MODEL = "gpt-4o-mini"  # TODO: 판정 모델 (gpt-4o-mini 권장)

# TODO: 프로젝트 도메인에 맞게 평가 차원을 수정하세요
EVAL_DIMENSIONS = [
    "정확성",    # TODO: 도메인에 맞게 수정
    "유용성",
    "관련성",
    "완결성",
    "안전성",
]


def llm_judge_evaluate(question, prediction, reference, dimensions):
    """
    LLM-as-a-Judge로 모델 출력을 평가합니다.
    
    Returns:
        dict: 각 차원별 점수 (1~5) 및 피드백
    """
    dimensions_text = "\n".join([f"- {d}: 1점(매우 나쁨)~5점(매우 좋음)" for d in dimensions])
    
    prompt = f"""당신은 AI 모델의 출력 품질을 평가하는 전문 심사위원입니다.

아래의 질문, 참고 답변, 모델 출력을 비교하여 평가해주세요.

## 질문
{question}

## 참고 답변 (정답)
{reference}

## 모델 출력 (평가 대상)
{prediction}

## 평가 기준
{dimensions_text}

다음 JSON 형식으로 평가해주세요:
{{
    "scores": {{{', '.join([f'"{d}": <1-5>' for d in dimensions])}}},
    "average": <평균 점수>,
    "feedback": "<종합 피드백>",
    "strengths": "<장점>",
    "weaknesses": "<개선점>"
}}"""
    
    try:
        response = client.chat.completions.create(
            model=JUDGE_MODEL,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.1,
            max_tokens=1000,
        )
        
        content = response.choices[0].message.content
        # JSON 파싱
        import re
        json_match = re.search(r"\{.*\}", content, re.DOTALL)
        if json_match:
            return json.loads(json_match.group())
    except Exception as e:
        print(f"❌ 평가 오류: {e}")
    
    return None


print("✅ LLM Judge 함수 정의 완료")

In [ ]:
# =====================================================
# 🔧 LLM-as-a-Judge 평가 실행
# TODO: 평가할 샘플 수를 조정하세요
# =====================================================

NUM_JUDGE_SAMPLES = 20  # TODO: 평가할 샘플 수 (API 비용 고려)

import random
random.seed(42)

# 샘플 선택
sample_indices = random.sample(range(len(eval_data)), min(NUM_JUDGE_SAMPLES, len(eval_data)))

judge_results = []

for idx in tqdm(sample_indices, desc="LLM Judge 평가"):
    item = eval_data[idx]
    prompt = item["instruction"]
    if item.get("input"):
        prompt += f"\n{item['input']}"
    
    result = llm_judge_evaluate(
        question=prompt,
        prediction=predictions[idx],
        reference=references[idx],
        dimensions=EVAL_DIMENSIONS,
    )
    
    if result:
        result["index"] = idx
        result["question"] = prompt[:100]
        judge_results.append(result)
    
    time.sleep(0.5)  # Rate limit

# 결과 집계
if judge_results:
    print(f"\n{'='*50}")
    print(f"🧑‍⚖️ LLM-as-a-Judge 평가 결과 ({len(judge_results)}개 샘플)")
    print(f"{'='*50}")
    
    # 차원별 평균
    for dim in EVAL_DIMENSIONS:
        scores = [r["scores"].get(dim, 0) for r in judge_results if "scores" in r]
        if scores:
            print(f"  {dim}: {np.mean(scores):.2f}/5.0 (±{np.std(scores):.2f})")
    
    # 전체 평균
    averages = [r.get("average", 0) for r in judge_results]
    print(f"\n  📊 종합 평균: {np.mean(averages):.2f}/5.0")
else:
    print("⚠️ 평가 결과가 없습니다. API 키를 확인해주세요.")

---
## 4️⃣ 정성 평가 (샘플 출력 검토)

모델 출력을 직접 확인하며 품질을 평가합니다.

In [ ]:
# =====================================================
# 👀 정성 평가: 샘플 출력 검토
# TODO: 직접 출력을 확인하고 코멘트를 남기세요
# =====================================================

NUM_REVIEW_SAMPLES = 10  # TODO: 검토할 샘플 수

review_indices = random.sample(range(len(eval_data)), min(NUM_REVIEW_SAMPLES, len(eval_data)))

reviews = []

for idx in review_indices:
    item = eval_data[idx]
    prompt = item["instruction"]
    if item.get("input"):
        prompt += f"\n{item['input']}"
    
    print(f"\n{'='*60}")
    print(f"📌 샘플 #{idx}")
    print(f"{'='*60}")
    print(f"\n❓ 질문: {prompt}")
    print(f"\n📗 정답: {references[idx][:300]}")
    print(f"\n🤖 모델: {predictions[idx][:300]}")
    print(f"\n---")
    
    # TODO: 각 샘플에 대해 코멘트를 남기세요
    reviews.append({
        "index": idx,
        "question": prompt[:100],
        "score": 0,      # TODO: 1~5 점수 부여
        "comment": "",   # TODO: 코멘트 작성
    })

print(f"\n⚠️ 위 reviews 리스트에 score와 comment를 채워주세요.")

---
## 5️⃣ 개선 전략 수립

평가 결과를 바탕으로 개선 전략을 수립합니다.

### 일반적인 개선 방향

| 문제 | 원인 | 개선 방법 |
|------|------|----------|
| 정확도 낮음 | 학습 데이터 품질 | 데이터 정제, 검수 강화 |
| 환각 발생 | 데이터 부족/불일치 | 데이터 증강, Fact 기반 데이터 추가 |
| 답변 짧음 | 학습 데이터 길이 분포 | 긴 답변 데이터 추가 |
| 반복 출력 | 학습 과다 | 에포크 줄이기, repetition_penalty |
| 형식 불일치 | 포맷 학습 부족 | 일관된 형식의 데이터 추가 |
| 특정 주제 약함 | 해당 주제 데이터 부족 | 주제별 데이터 보강 |

In [ ]:
# =====================================================
# 📝 개선 전략 수립 템플릿
# TODO: 평가 결과를 분석하고 전략을 작성하세요
# =====================================================

improvement_plan = {
    # 현재 성능 요약
    "current_performance": {
        "rouge_l": np.mean(rouge_scores["rougeL"]) if rouge_scores["rougeL"] else 0,
        "bleu": np.mean(bleu_scores) if bleu_scores else 0,
        "llm_judge_avg": np.mean(averages) if judge_results else 0,
    },
    
    # TODO: 주요 발견사항 작성
    "findings": [
        "",  # TODO: 발견사항 1 (예: "의학 용어 정확성 부족")
        "",  # TODO: 발견사항 2
        "",  # TODO: 발견사항 3
    ],
    
    # TODO: 개선 전략 작성
    "strategies": [
        {
            "action": "",         # TODO: 개선 행동 (예: "전문 용어 데이터 100개 추가")
            "expected_impact": "", # TODO: 기대 효과
            "priority": "",       # TODO: "high", "medium", "low"
        },
        # TODO: 추가 전략
    ],
    
    # TODO: 하이퍼파라미터 변경 사항
    "hp_changes": {
        "learning_rate": None,    # TODO: 변경할 학습률 (예: 1e-4)
        "epochs": None,           # TODO: 변경할 에포크 수
        "lora_r": None,           # TODO: 변경할 LoRA rank
    },
}

# 전략 출력
print("📋 개선 전략")
print("="*50)
print(f"\n현재 성능:")
for k, v in improvement_plan["current_performance"].items():
    print(f"  {k}: {v:.4f}")
print(f"\n발견사항:")
for f in improvement_plan["findings"]:
    if f:
        print(f"  - {f}")
print(f"\n개선 전략:")
for s in improvement_plan["strategies"]:
    if s["action"]:
        print(f"  - [{s['priority']}] {s['action']} → {s['expected_impact']}")

---
## 6️⃣ 반복 학습 및 비교

개선 전략을 적용하여 **재학습**하고, 이전 모델과 성능을 비교합니다.

> 💡 **Hint**: 이 섹션은 34_project_training.ipynb로 돌아가서 재학습한 후, 다시 이 노트북에서 평가합니다.
> 또는 아래 코드를 사용하여 간단히 재학습할 수 있습니다.

In [ ]:
# =====================================================
# 🔄 모델 버전 비교 템플릿
# TODO: 여러 버전의 모델을 비교하세요
# =====================================================

# 이전 학습 결과를 기록합니다
# TODO: 각 반복 학습 후 결과를 추가하세요

experiment_log = [
    {
        "version": "v1",
        "description": "초기 학습",  # TODO: 설명
        "changes": "기본 설정",
        "rouge_l": np.mean(rouge_scores["rougeL"]) if rouge_scores["rougeL"] else 0,
        "bleu": np.mean(bleu_scores) if bleu_scores else 0,
        "llm_judge": np.mean(averages) if judge_results else 0,
    },
    # TODO: v2, v3 등 추가 실험 결과를 여기에 추가
    # {
    #     "version": "v2",
    #     "description": "데이터 200개 추가",
    #     "changes": "데이터 증강",
    #     "rouge_l": 0.0,
    #     "bleu": 0.0,
    #     "llm_judge": 0.0,
    # },
]

# 비교 테이블 출력
if experiment_log:
    df = pd.DataFrame(experiment_log)
    print("📊 실험 비교")
    print(df.to_string(index=False))

---
## 7️⃣ 최종 성능 보고서

프로젝트의 **최종 성능 보고서**를 작성합니다.

In [ ]:
# =====================================================
# 📋 최종 성능 보고서 생성
# TODO: 빈 항목을 채워주세요
# =====================================================

final_report = {
    "project_name": project_plan["project_name"],
    "model": MODEL_NAME,
    "method": METHOD,
    "data_count": {
        "train": len(eval_data),  # 실제 train 수로 교체
        "eval": len(eval_data),
    },
    
    # 자동 메트릭
    "auto_metrics": {
        "rouge1": float(np.mean(rouge_scores["rouge1"])) if rouge_scores["rouge1"] else 0,
        "rouge2": float(np.mean(rouge_scores["rouge2"])) if rouge_scores["rouge2"] else 0,
        "rougeL": float(np.mean(rouge_scores["rougeL"])) if rouge_scores["rougeL"] else 0,
        "bleu": float(np.mean(bleu_scores)) if bleu_scores else 0,
    },
    
    # LLM Judge
    "llm_judge": {
        "overall": float(np.mean(averages)) if judge_results else 0,
        "per_dimension": {},
    },
    
    # TODO: 정성 평가 요약
    "qualitative_summary": "",    # TODO: 정성 평가 요약
    
    # TODO: 주요 개선 사항
    "improvements_made": [],       # TODO: 수행한 개선 사항
    
    # TODO: 한계점 및 향후 과제
    "limitations": [],             # TODO: 한계점
    "future_work": [],             # TODO: 향후 과제
}

# LLM Judge 차원별 점수 추가
if judge_results:
    for dim in EVAL_DIMENSIONS:
        scores = [r["scores"].get(dim, 0) for r in judge_results if "scores" in r]
        if scores:
            final_report["llm_judge"]["per_dimension"][dim] = float(np.mean(scores))

# 보고서 출력
print("\n" + "="*60)
print(f"📋 최종 성능 보고서: {final_report['project_name']}")
print("="*60)
print(f"\n🤖 모델: {final_report['model']} ({final_report['method']})")
print(f"\n📊 자동 평가 메트릭:")
for k, v in final_report["auto_metrics"].items():
    print(f"   {k}: {v:.4f}")
print(f"\n🧑‍⚖️ LLM-as-a-Judge:")
print(f"   종합: {final_report['llm_judge']['overall']:.2f}/5.0")
for dim, score in final_report["llm_judge"]["per_dimension"].items():
    print(f"   {dim}: {score:.2f}/5.0")

# 보고서 저장
report_path = f"{PROJECT_DIR}/results/final_report.json"
os.makedirs(f"{PROJECT_DIR}/results", exist_ok=True)
with open(report_path, "w", encoding="utf-8") as f:
    json.dump(final_report, f, ensure_ascii=False, indent=2)

print(f"\n💾 보고서 저장: {report_path}")

---
## 📝 정리

### 이번 세션에서 완료한 것
- ✅ 평가 데이터셋 기반 모델 응답 생성
- ✅ 자동 평가 메트릭 계산 (ROUGE, BLEU)
- ✅ LLM-as-a-Judge 다차원 평가
- ✅ 정성 평가 (샘플 출력 검토)
- ✅ 개선 전략 수립
- ✅ 반복 학습 및 버전 비교
- ✅ 최종 성능 보고서 작성

### 산출물
- 📁 `my_project/results/final_report.json` - 최종 성능 보고서

### 다음 노트북
👉 **36_project_deployment.ipynb**: 프로젝트 배포 및 어플리케이션 구현